# 🌿 Luminos Skin LP — DALL·E 3 画像生成

このノートブックで以下の5枚を自動生成できます。

| ファイル名 | 内容 | 品質 | 費用 |
|-----------|------|------|------|
| fv-hero.jpg | FVメイン施術シーン（縦長） | HD | $0.08 |
| menu-01.jpg | ベーシックフェイシャル | Standard | $0.04 |
| menu-02.jpg | エイジングケア | Standard | $0.04 |
| menu-03.jpg | 保湿集中フェイシャル | Standard | $0.04 |
| owner.jpg | オーナー写真（縦長） | HD | $0.08 |

**合計推定コスト: 約 $0.28（約40円）**

---
### 手順
1. 上から順にセルを実行（▶ ボタン）
2. Step 2 でAPIキーを入力
3. Step 3 で画像生成（5〜10分かかります）
4. Step 4 でZIPダウンロード → GitHubにアップロード

In [ ]:
# ====================================================
# Step 1: パッケージインストール
# ====================================================
!pip install openai -q
import openai, os, time, zipfile, getpass
from pathlib import Path
from IPython.display import display, Image as IPImage

OUTPUT_DIR = Path("luminos-skin-images")
OUTPUT_DIR.mkdir(exist_ok=True)
print("✅ 準備完了")

In [ ]:
# ====================================================
# Step 2: OpenAI APIキーを入力
# （入力欄に貼り付けて Enter → 画面には表示されません）
# ====================================================
api_key = getpass.getpass("OpenAI APIキー (sk-...): ")
client = openai.OpenAI(api_key=api_key)

# 接続テスト
try:
    client.models.list()
    print("✅ APIキー確認OK")
except Exception as e:
    print(f"❌ エラー: {e}")
    print("APIキーを確認してください: https://platform.openai.com/api-keys")

In [ ]:
# ====================================================
# Step 3: 画像生成（5〜10分かかります）
# ====================================================

IMAGES = [
    {
        "filename": "fv-hero.jpg",
        "size": "1024x1792",
        "quality": "hd",
        "desc": "FVヒーロー — 施術シーン（縦長）",
        "prompt": (
            "A 35-year-old Japanese woman with clear, radiant skin lying on a white facial treatment bed "
            "in a luxury private salon room. Warm natural window light from the left. Beige and cream interior. "
            "Professional female esthetician's hands gently applying facial massage. "
            "The client's expression is serene and relaxed, eyes closed. "
            "Shot from slightly above at a 45-degree angle. Very shallow depth of field. "
            "No text, no logos, no watermarks. "
            "High-end Japanese beauty salon photography. Film-like warm tone."
        ),
    },
    {
        "filename": "menu-01.jpg",
        "size": "1792x1024",
        "quality": "standard",
        "desc": "メニュー01 — ベーシックフェイシャル",
        "prompt": (
            "Close-up professional photo of a facial cleansing and exfoliation treatment "
            "on a Japanese woman's face in a private luxury esthetic salon. "
            "Soft warm overhead lighting. Clean white towels, professional skincare tools visible. "
            "Calm and serene atmosphere. No text, no logos. "
            "High-end beauty photography, warm neutral tones."
        ),
    },
    {
        "filename": "menu-02.jpg",
        "size": "1792x1024",
        "quality": "standard",
        "desc": "メニュー02 — エイジングケア",
        "prompt": (
            "Professional photo of a face lifting and contouring massage treatment — "
            "female esthetician's hands performing gentle upward strokes on a Japanese woman's "
            "jawline and cheeks in a luxurious private esthetic room. "
            "Soft warm lighting from above. Beige interior. "
            "Professional and serene atmosphere. No text, no logos."
        ),
    },
    {
        "filename": "menu-03.jpg",
        "size": "1792x1024",
        "quality": "standard",
        "desc": "メニュー03 — 保湿集中フェイシャル",
        "prompt": (
            "Professional photo of a deep hydration face mask treatment — "
            "a Japanese woman wearing a cream-colored moisturizing sheet mask, "
            "lying relaxed on a white treatment bed in a luxury private salon. "
            "Warm soft lighting from above. White sheets, small candles or plants nearby. "
            "Rejuvenating, serene atmosphere. No text, no logos."
        ),
    },
    {
        "filename": "owner.jpg",
        "size": "1024x1792",
        "quality": "hd",
        "desc": "オーナー写真（縦長）",
        "prompt": (
            "Professional portrait of a 38-year-old Japanese female esthetician and salon owner. "
            "Wearing a clean white or light beige esthetic uniform. "
            "Gentle, warm, confident smile. Trustworthy and professional expression. "
            "Simple soft beige or cream background. Soft studio lighting from the front. "
            "Shot from waist level. Professional beauty business headshot style. "
            "No text, no logos, no watermarks."
        ),
    },
]

import urllib.request

for i, img in enumerate(IMAGES, 1):
    out_path = OUTPUT_DIR / img["filename"]
    if out_path.exists():
        print(f"  [{i}/{len(IMAGES)}] スキップ（既存）: {img['filename']}")
        continue

    print(f"  [{i}/{len(IMAGES)}] 生成中: {img['filename']} ({img['quality']})")
    print(f"         {img['desc']}")

    try:
        response = client.images.generate(
            model="dall-e-3",
            prompt=img["prompt"],
            n=1,
            size=img["size"],
            quality=img["quality"],
            response_format="url",
        )
        img_url = response.data[0].url
        urllib.request.urlretrieve(img_url, out_path)
        print(f"         ✅ 保存完了")

        # プレビュー表示
        display(IPImage(filename=str(out_path), width=300))

    except Exception as e:
        print(f"         ❌ エラー: {e}")

    if i < len(IMAGES):
        print("         （次の画像まで5秒待機...）")
        time.sleep(5)

print("\n🎉 全画像の生成が完了しました！")

In [ ]:
# ====================================================
# Step 4: ZIPファイルにまとめてダウンロード
# ====================================================
from google.colab import files

zip_path = "luminos-skin-images.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for f in OUTPUT_DIR.iterdir():
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
            zf.write(f, f.name)
            print(f"  + {f.name} ({f.stat().st_size // 1024} KB)")

print(f"\n📦 ZIPファイル作成完了: {zip_path}")
files.download(zip_path)
print("✅ ダウンロード開始しました")

---
## ダウンロード後の手順

1. **ZIPを解凍**する
2. **GitHubにアップロード**:
   - リポジトリの `beauty-salon-lp/images/` フォルダを開く
   - 「Add file」→「Upload files」
   - 解凍した5枚の画像をドラッグ＆ドロップ
   - 「Commit changes」ボタンをクリック
3. **Claudeに報告**: 「画像をアップロードしました」と伝えると、HTMLに組み込みます！

---
### ⚠️ 画像の確認ポイント（薬機法）
- 手・指の本数が自然か（5本）
- 顔の造形が崩れていないか
- 意図しないテキスト・ロゴが入っていないか
- 施術効果を過度に示唆していないか